# Dispersion-per-color diagnostics -- Simbad-confirmed stable stars

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-17

## Goal

`11_Dispersionperband_diagnostics.ipynb` studies the *per-band* photometric
scatter `mmag_meas` for the Simbad-confirmed stable-star sample. This
notebook extends the same diagnostic to **colors**: `u-g`, `g-r`, `r-i`,
`i-z`, `z-y`.

Colors cannot be read off a single visit: the two bands of a color pair are
never observed simultaneously, so a "color" is only ever an estimate built
from **two different visits**, separated by some time gap. Since PWV and
aerosol content vary with time, a color built from two visits far apart in
time is a noisier, less physical estimate of the star's intrinsic color than
one built from two visits close together in time. This notebook therefore:

1. For each star and each color pair `(b1, b2)`, pairs every `b1` visit with
   its **nearest-in-time** `b2` visit (nearest-neighbor match on
   `expMidptMJD`, within a configurable `MAX_DT_DAYS` tolerance), and records
   the time gap `dt_days = |mjd_b1 - mjd_b2|` alongside each resulting color
   estimate.
2. Computes, per star and color pair, a **weighted median color**, where each
   paired estimate is weighted by a decreasing function of `dt_days` (short
   time gap -> more weight), and a matching **weighted robust dispersion**
   (`sigma_IQR` computed on the weighted distribution).
3. Plots, per color pair, the deviation of every individual paired color
   estimate from its star's weighted median (in mmag) against that star's
   weighted median color (in mag), with each point's color mapped to
   `dt_days` and its visual prominence (opacity + draw order) increasing as
   `dt_days` shrinks -- so the pairs that best approximate a true
   simultaneous color stand out.

## Input data

Same per-visit light-curve table as `11_Dispersionperband_diagnostics.ipynb`,
written by `02_MergeLCsourceswithMJD.ipynb`:
`data_MergeVisits_02_out/all_stars_lightcurves_mjd.csv`, one row per (star,
visit), with columns including `simbad_id`, `band`, `psfFlux`, `psfFluxErr`,
`expMidptMJD`, `ra`, `dec`.


## 1. Imports

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

from astropy.time import Time
from scipy.optimize import curve_fit

In [ ]:
# view all contents of pandas tables
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging is configured and working in the notebook!")

## 3. Configuration

**Edit only this cell** to point to the input light-curve file (same file
read by `11_Dispersionperband_diagnostics.ipynb`) and to adjust the
color-pairing / weighting parameters used below.

In [ ]:
# -- Notebook tag ------------------------------------------------------------
NB_TAG = "DispersionPerColor_12"

# -- Input: per-visit merged LC file from notebook 02 ------------------------
DIR_DATA_IN = "./data_MergeVisits_02_out"
LC_CSV = os.path.join(DIR_DATA_IN, "all_stars_lightcurves_mjd.csv")

# -- Output figures ------------------------------------------------------------
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# -- Photometric columns in the per-visit table -------------------------------
FLUX_COL = "psfFlux"
FLUX_ERR_COL = "psfFluxErr"
BAND_COL = "band"
STAR_COL = "simbad_id"
MJD_COL = "expMidptMJD"
DAY_OBS_COL = "day_obs"  # LSST "night" identifier, YYYYMMDD, one value per visit

# -- AB magnitude zero point for fluxes expressed in nJy ----------------------
ABMAG_ZP_NJY = 31.4

# -- All LSST bands, in the standard display order ----------------------------
BANDS = ["u", "g", "r", "i", "z", "y"]
BANDS_COLOR = {
    "u": "blueviolet",
    "g": "limegreen",
    "r": "red",
    "i": "darkorange",
    "z": "chocolate",
    "y": "saddlebrown",
}

# -- Color pairs studied here, adjacent bands only, in the standard order -----
COLOR_PAIRS = [("u", "g"), ("g", "r"), ("r", "i"), ("i", "z"), ("z", "y")]
COLOR_NAMES = [f"{b1}-{b2}" for b1, b2 in COLOR_PAIRS]

# -- Color-pairing tolerance: a (b1, b2) visit pair is only kept if the two
#    visits are separated by less than MAX_DT_DAYS. Beyond that, atmospheric
#    conditions (and, in principle, intrinsic variability) may have changed
#    too much for the pair to still approximate a "simultaneous" color.
MAX_DT_DAYS = 2.0

# -- Weighting of individual (b1, b2) color estimates when combining them
#    into one weighted median color per star: weight = exp(-dt_days / DT_WEIGHT_SCALE_DAYS),
#    so pairs separated by less than ~DT_WEIGHT_SCALE_DAYS dominate the median
#    and more distant pairs contribute comparatively little (but are not
#    discarded outright, unlike the hard MAX_DT_DAYS cut above).
DT_WEIGHT_SCALE_DAYS = 0.5

# -- Minimum number of good (b1, b2) pairs to keep a (star, color) row -------
MIN_PAIRS_PER_STARCOLOR = 5

# -- Visual prominence of a point in the 8.2 scatter plots, as a function of
#    dt_days: alpha = ALPHA_FLOOR + (1 - ALPHA_FLOOR) * exp(-dt_days / DT_ALPHA_SCALE_DAYS),
#    so pairs close in time stand out (alpha -> 1) and distant pairs fade out
#    (alpha -> ALPHA_FLOOR) without disappearing completely.
DT_ALPHA_SCALE_DAYS = 0.5
ALPHA_FLOOR = 0.08

# -- Colormap used to encode dt_days (point color) in the 8.2/8.3 figures ----
DT_CMAP = "plasma"

# -- Section 8.3: near-simultaneous cut used to look for a genuine trend of
#    |color deviation| with the star's median color (a real color-dependent
#    systematic, not a dt-pairing artifact, should only show up once dt_days
#    is small).
DT_STRICT_DAYS = 0.5
N_BINS_2DHIST = 40  # 2D histogram resolution (median color x |deviation|)
N_COLOR_PROFILE_BINS = 20  # number of running-profile bins over median color
MIN_PTS_PROFILE_BIN = 3  # minimum pairs in a profile bin to plot a point
RANGE_HIST2D = [[0.2, 1.5], [0, 60]]

# -- Section 8.6: night-by-night time dependence of the color dispersion ----
# Stars are split into N_COLOR_SLICES tertiles of their own median color so a
# genuine night-to-night effect (e.g. driven by PWV or aerosol content on a
# given night) can be told apart from a color-slice effect. A (night, slice)
# point is only plotted once it has at least MIN_PAIRS_PER_NIGHT_SLICE pairs.
N_COLOR_SLICES = 3
MIN_PAIRS_PER_NIGHT_SLICE = 3

log.info(f"Reading per-visit light curves from '{LC_CSV}'")
log.info(f"Color pairs: {COLOR_NAMES}")

## 4. Helper functions

In [ ]:
# -- savefig: PDF + PNG -------------------------------------------------------
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
# -- helpers for statistics ----------------------------------------------------
def sigma_iqr(x):
    """Robust scatter estimator: interquartile range rescaled so that it
    matches the standard deviation for a pure Gaussian distribution."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    q25, q75 = np.percentile(x, [25, 75])
    return (q75 - q25) / 1.3489795


def flux_to_mag(flux_njy):
    """AB magnitude from a flux expressed in nJy, using ABMAG_ZP_NJY."""
    flux_njy = np.asarray(flux_njy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = ABMAG_ZP_NJY - 2.5 * np.log10(flux_njy)
    return mag


def weighted_quantile(values, weights, quantiles):
    """Weighted quantiles of `values` (weights `weights`, same length).

    `quantiles` is an array-like of quantiles in [0, 1]. Uses linear
    interpolation on the weighted empirical CDF (values sorted ascending).
    Returns an array the same length as `quantiles`, filled with NaN if
    there is no finite, positively-weighted data.
    """
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    quantiles = np.atleast_1d(np.asarray(quantiles, dtype=float))

    sel = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[sel], weights[sel]
    if len(values) == 0:
        return np.full(len(quantiles), np.nan)

    order = np.argsort(values)
    values, weights = values[order], weights[order]

    # midpoint cumulative-weight CDF: cw[i] is the CDF value at values[i]
    cw = np.cumsum(weights) - 0.5 * weights
    cw /= np.sum(weights)

    return np.interp(quantiles, cw, values)


def weighted_median(values, weights):
    """Weighted median of `values`, weighted by `weights`."""
    return weighted_quantile(values, weights, [0.5])[0]


def weighted_sigma_iqr(values, weights):
    """Weighted robust scatter estimator: weighted interquartile range,
    rescaled to match the standard deviation for a pure Gaussian."""
    q25, q75 = weighted_quantile(values, weights, [0.25, 0.75])
    if not np.isfinite(q25) or not np.isfinite(q75):
        return np.nan
    return (q75 - q25) / 1.3489795


def dt_weight(dt_days, scale_days=DT_WEIGHT_SCALE_DAYS):
    """Weight given to a color estimate separated by `dt_days`: an
    exponential decay with scale `scale_days`, so pairs well within
    `scale_days` dominate and more distant pairs contribute little."""
    dt_days = np.asarray(dt_days, dtype=float)
    return np.exp(-dt_days / scale_days)

In [ ]:
# -- helper for color pairing --------------------------------------------------
def match_color_pairs(df_star, band1, band2, max_dt_days=MAX_DT_DAYS):
    """For a single star's per-visit table `df_star`, pair every `band1`
    visit with its nearest-in-time `band2` visit (nearest-neighbor match on
    MJD_COL), keeping only pairs separated by less than `max_dt_days`.

    Returns a DataFrame with one row per matched pair: `mjd1`, `mjd2`,
    `mjd_mean`, `dt_days`, `mag1`, `mag2`, `color` (= mag1 - mag2), and
    `day_obs1`, `day_obs2` (the LSST "night" identifier, YYYYMMDD, of each of
    the two visits -- kept so any excess dispersion that turns out to
    correlate with time can be traced back to the individual visits, and so
    pairs can be grouped by observing night, e.g. in section 8.6). Empty (but
    correctly-columned) if either band has no valid visits for this star.
    """
    cols = ["mjd1", "mjd2", "mjd_mean", "dt_days", "mag1", "mag2", "color", "day_obs1", "day_obs2"]

    sub1 = df_star.loc[df_star[BAND_COL] == band1, [MJD_COL, FLUX_COL, DAY_OBS_COL]].dropna(
        subset=[MJD_COL, FLUX_COL]
    )
    sub2 = df_star.loc[df_star[BAND_COL] == band2, [MJD_COL, FLUX_COL, DAY_OBS_COL]].dropna(
        subset=[MJD_COL, FLUX_COL]
    )
    sub1 = sub1.loc[sub1[FLUX_COL] > 0].sort_values(MJD_COL)
    sub2 = sub2.loc[sub2[FLUX_COL] > 0].sort_values(MJD_COL)

    if len(sub1) == 0 or len(sub2) == 0:
        return pd.DataFrame(columns=cols)

    sub1 = sub1.rename(columns={MJD_COL: "mjd1", FLUX_COL: "flux1", DAY_OBS_COL: "day_obs1"})
    sub2 = sub2.rename(columns={MJD_COL: "mjd2", FLUX_COL: "flux2", DAY_OBS_COL: "day_obs2"})

    merged = pd.merge_asof(
        sub1,
        sub2,
        left_on="mjd1",
        right_on="mjd2",
        direction="nearest",
        tolerance=max_dt_days,
    )
    merged = merged.dropna(subset=["mjd2"])  # visits with no match inside the tolerance
    if len(merged) == 0:
        return pd.DataFrame(columns=cols)

    merged["dt_days"] = (merged["mjd1"] - merged["mjd2"]).abs()
    merged["mjd_mean"] = 0.5 * (merged["mjd1"] + merged["mjd2"])
    merged["mag1"] = flux_to_mag(merged["flux1"].to_numpy())
    merged["mag2"] = flux_to_mag(merged["flux2"].to_numpy())
    merged["color"] = merged["mag1"] - merged["mag2"]

    return merged[cols]

## 5. Read the per-visit light curves

In [ ]:
df_lc = pd.read_csv(LC_CSV)
log.info(f"Read {len(df_lc)} per-visit rows for {df_lc[STAR_COL].nunique()} stars from {LC_CSV}")
df_lc.head()

## 6. Build the per-(star, visit-pair) color table

For every star and every color pair `(b1, b2)`, pair up the visits with
`match_color_pairs` and stack the results into a single long table,
`df_color_pairs`, with one row per **individual color estimate** (i.e. per
matched visit pair), tagged with `object_id` and `color_name`.

`match_color_pairs` keeps `mjd1`, `mjd2` (visit epochs), `mag1`, `mag2` (the two per-visit magnitudes that make up the color) and `day_obs1`, `day_obs2` (the LSST night identifier, `YYYYMMDD`, for each visit) on every row of `df_color_pairs`. This is what lets section 8.6 group individual color estimates by observing night and check for a time dependence of the dispersion.

In [ ]:
rows = []

# loop on stars
for sid, df_star in df_lc.groupby(STAR_COL):
    # loop on color pair
    for (b1, b2), color_name in zip(COLOR_PAIRS, COLOR_NAMES):
        pairs = match_color_pairs(df_star, b1, b2)
        if len(pairs) == 0:
            continue
        pairs = pairs.copy()
        pairs["object_id"] = sid
        pairs["color_name"] = color_name
        rows.append(pairs)

# dataframe containing all pairs
df_color_pairs = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
log.info(
    f"Built {len(df_color_pairs)} individual color estimates "
    f"for {df_color_pairs['object_id'].nunique() if len(df_color_pairs) else 0} stars "
    f"across {len(COLOR_NAMES)} color pairs (MAX_DT_DAYS={MAX_DT_DAYS})."
)
df_color_pairs.head()

In [ ]:
df_color_pairs.groupby("color_name")["dt_days"].describe()[["count", "mean", "50%", "std"]].reindex(
    COLOR_NAMES
)

## 7. Weighted per-(star, color) median and dispersion

For each `(object_id, color_name)` group, combine the individual color
estimates into a single **weighted median color** (`weight = dt_weight(dt_days)`,
see Configuration) and a matching **weighted robust dispersion**
(`weighted_sigma_iqr`). Only `(star, color)` groups with at least
`MIN_PAIRS_PER_STARCOLOR` estimates are kept.

The weighted median is then merged back onto `df_color_pairs` so that every
individual estimate carries its own star's weighted-median color alongside
it, letting us compute a per-estimate deviation from that median.

In [ ]:
def _summarize_group(g):
    w = dt_weight(g["dt_days"].to_numpy())
    color_vals = g["color"].to_numpy()
    return pd.Series(
        {
            "n_pairs": len(g),
            "color_median_w": weighted_median(color_vals, w),
            "sigma_iqr_w": weighted_sigma_iqr(color_vals, w),
            "dt_days_median": np.median(g["dt_days"].to_numpy()),
            "dt_days_min": np.min(g["dt_days"].to_numpy()),
        }
    )


df_color_summary = (
    df_color_pairs.groupby(["object_id", "color_name"], group_keys=True)
    .apply(_summarize_group, include_groups=False)
    .reset_index()
)
df_color_summary = df_color_summary.loc[df_color_summary["n_pairs"] >= MIN_PAIRS_PER_STARCOLOR].copy()

# convert weighted sigma to mmag for readability (colors themselves stay in mag)
df_color_summary["sigma_iqr_w_mmag"] = 1000.0 * df_color_summary["sigma_iqr_w"]

log.info(
    f"Built weighted per-(star, color) summary: {len(df_color_summary)} rows "
    f">= {MIN_PAIRS_PER_STARCOLOR} pairs/(star,color)."
)
df_color_summary.head()

In [ ]:
df_color_summary.groupby("color_name")["sigma_iqr_w_mmag"].describe()[
    ["count", "mean", "50%", "std"]
].reindex(COLOR_NAMES)

In [ ]:
# -- Merge the weighted median color back onto the individual estimates, and
# derive the per-estimate deviation from it (in mmag, consistent with the
# rest of the pipeline's mmag convention for dispersions).
df_color_pairs = df_color_pairs.merge(
    df_color_summary[["object_id", "color_name", "color_median_w", "n_pairs"]],
    on=["object_id", "color_name"],
    how="inner",  # drops estimates whose (star, color) group didn't pass MIN_PAIRS_PER_STARCOLOR
)
df_color_pairs["color_dev_mmag"] = 1000.0 * (df_color_pairs["color"] - df_color_pairs["color_median_w"])

log.info(f"{len(df_color_pairs)} individual color estimates retained after the MIN_PAIRS_PER_STARCOLOR cut.")

## 8. Plots

### 8.1  Weighted robust color dispersion per color pair -- boxplot

Boxplot of the weighted `sigma_IQR` (mmag), one box per color pair, in the
order `u-g, g-r, r-i, i-z, z-y`. One point per star per color (from
`df_color_summary`). Analogous to section 6.1 of
`11_Dispersionperband_diagnostics.ipynb`, but for colors instead of bands.
If the dispersion is atmospherically driven the way `11_...ipynb` suggests
for the bluest/reddest bands, `u-g` and `z-y` are the color pairs most
likely to show excess scatter (aerosols pull on `u-g` through the `u` band,
PWV pulls on `z-y` through both `z` and `y`).

In [ ]:
YMAX = 70.0
YMIN = 0.0

data = [
    df_color_summary.loc[df_color_summary["color_name"] == cn, "sigma_iqr_w_mmag"].dropna().to_numpy()
    for cn in COLOR_NAMES
]

fig, ax = plt.subplots(figsize=(8, 5))

flier_props = dict(
    marker="o",
    markersize=6,
    markerfacecolor="none",
    markeredgecolor="darkgray",
    markeredgewidth=1.0,
    alpha=0.6,
)

# show the boxplot
ax.boxplot(data, tick_labels=COLOR_NAMES, showfliers=True, flierprops=flier_props)

ax.set_xlabel("Color")
ax.set_ylabel(r"weighted $\sigma_{IQR}$ (mmag)")
ax.set_title("Weighted robust color dispersion, Simbad-confirmed stable stars")
ax.grid(True, alpha=0.3)

for idx, d in enumerate(data, start=1):
    ax.text(
        idx,
        0.95,
        f"n={len(d)}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9,
        color="darkgray",
    )
ax.set_ylim(YMIN, YMAX)
plt.tight_layout()
savefig(fig, "colordisp_stablestars_perpair_boxplot")
plt.show()

### 8.2  Color-estimate deviation vs. star median color, colored by `dt_days`

The figure requested: `1 x 5` grid, one panel per color pair (order `u-g,
g-r, r-i, i-z, z-y`). Each point is one individual `(b1, b2)` visit pairing
for one star:

- **x** = that star's weighted-median color (mag) -- so all pairings of the
  same star sit at the same x position within a panel.
- **y** = deviation of that pairing's color estimate from the star's
  weighted median (mmag).
- **point color** = `dt_days`, the time gap between the two visits used to
  build that particular color estimate (shared colormap/scale across all
  five panels).
- **point opacity + draw order** increase as `dt_days` shrinks, so pairs
  built from near-simultaneous visits (the most trustworthy color
  estimates) stand out over the more numerous, noisier, larger-`dt_days`
  pairs plotted underneath them.

In [ ]:
YMAX = 100.0
YMIN = -100.0

# shared color scale for dt_days across all five panels
_dt_all = df_color_pairs["dt_days"].to_numpy()
_dt_floor = max(np.nanmin(_dt_all[_dt_all > 0]) * 0.5, 1e-4) if np.any(_dt_all > 0) else 1e-4
dt_norm = LogNorm(vmin=_dt_floor, vmax=np.nanmax(_dt_all))

# create the figure
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)

mesh = None

# loop on colors
for ax, color_name in zip(axes, COLOR_NAMES):
    # extract the color pair
    sub = df_color_pairs.loc[df_color_pairs["color_name"] == color_name].copy()
    if len(sub) == 0:
        ax.set_title(f"{color_name} (no data)")
        continue

    # visibility increases as dt_days shrinks: alpha per point, and draw
    # short-dt points last (painter's algorithm -> they end up on top)
    alpha = ALPHA_FLOOR + (1.0 - ALPHA_FLOOR) * np.exp(-sub["dt_days"].to_numpy() / DT_ALPHA_SCALE_DAYS)
    sub = sub.assign(_alpha=alpha).sort_values("dt_days", ascending=False)

    # scatter plot
    mesh = ax.scatter(
        sub["color_median_w"],
        sub["color_dev_mmag"],
        c=sub["dt_days"],
        cmap=DT_CMAP,
        norm=dt_norm,
        s=22,
        alpha=sub["_alpha"].to_numpy(),
        edgecolor="none",
    )

    ax.axhline(0.0, color="gray", ls="--", lw=1.0, alpha=0.7)
    n_stars = sub["object_id"].nunique()
    ax.set_title(f"{color_name}  (N={len(sub)} pairs, {n_stars} stars)")
    ax.set_xlabel(f"median {color_name} (mag)")
    ax.grid(True, alpha=0.2)
    ax.set_ylim(YMIN, YMAX)

axes[0].set_ylabel("color deviation from star median (mmag)")

if mesh is not None:
    cbar = fig.colorbar(mesh, ax=axes, location="right", pad=0.01, fraction=0.02)
    cbar.set_label(r"$\Delta t$ (days) between the two visits used for the color")

fig.suptitle(
    "Color-estimate deviation vs. star median color, colored by visit time gap\n"
    "(brighter/opaque points = shorter time gap between the two bands' visits)",
    y=1.04,
    fontsize=15,
)
savefig(fig, "colordev_vs_mediancolor_scatter_by_dt")
plt.show()

### 8.3  Color-estimate absolute deviation vs. star median color (2D histograms)

Same `1 x 5` grid and pairs as section 8.2, but restricted to the
near-simultaneous estimates (`dt_days < DT_STRICT_DAYS`, same data as the
brightest points of 8.2) so that any trend seen here is a genuine
color-dependent systematic, not an artifact of poorly-paired visits. Each
panel is a 2D histogram of `|color deviation|` (mmag, y-axis) against the
star's weighted-median color (mag, x-axis). A red profile line overlays
the mean `|deviation|` in `N_COLOR_PROFILE_BINS` equal-population bins of
median color (error bars = standard error on that mean) -- the most direct
way to read off whether the dispersion increases with color.

In [ ]:
strict = df_color_pairs.loc[df_color_pairs["dt_days"] < DT_STRICT_DAYS].copy()
strict["abs_color_dev_mmag"] = strict["color_dev_mmag"].abs()

fig, axes = plt.subplots(1, 5, figsize=(26, 5), sharey=False)

hist_image = None
for ax, color_name in zip(axes, COLOR_NAMES):
    sub = strict.loc[strict["color_name"] == color_name]
    if len(sub) == 0:
        ax.set_title(f"{color_name} (no data)")
        continue

    x = sub["color_median_w"].to_numpy()
    y = sub["abs_color_dev_mmag"].to_numpy()

    # 2D histogram: median color (x) vs. |color deviation| (y)
    _, _, _, hist_image = ax.hist2d(x, y, bins=N_BINS_2DHIST, range=RANGE_HIST2D, cmap="viridis", cmin=1)

    # running profile: mean |deviation| in equal-population bins of median color
    edges = np.unique(np.quantile(x, np.linspace(0, 1, N_COLOR_PROFILE_BINS + 1)))
    if len(edges) > 2:
        bin_idx = np.digitize(x, edges[1:-1])
        prof_x, prof_y, prof_yerr = [], [], []
        for b in range(len(edges) - 1):
            m = bin_idx == b
            if m.sum() < MIN_PTS_PROFILE_BIN:
                continue
            prof_x.append(np.mean(x[m]))
            prof_y.append(np.mean(y[m]))
            prof_yerr.append(np.std(y[m], ddof=1) / np.sqrt(m.sum()) if m.sum() > 1 else 0.0)

        if prof_x:
            ax.errorbar(
                prof_x,
                prof_y,
                yerr=prof_yerr,
                fmt="o-",
                color="crimson",
                ms=4,
                lw=1.5,
                capsize=2,
                label="mean |dev| (SEM)",
            )
            ax.legend(fontsize=8, loc="upper left")

    n_stars = sub["object_id"].nunique()
    ax.set_title(f"{color_name}  (N={len(sub)} pairs, {n_stars} stars, dt<{DT_STRICT_DAYS:.1f}d)")
    ax.set_xlabel(f"median {color_name} (mag)")
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel("|color deviation| from star median (mmag)")

if hist_image is not None:
    cbar = fig.colorbar(hist_image, ax=axes, location="right", pad=0.01, fraction=0.02)
    cbar.set_label("N pairs per bin")

fig.suptitle(
    f"Absolute color-estimate deviation vs. star median color (dt_days < {DT_STRICT_DAYS} d)\n"
    "red line = mean |deviation| per color bin, error bars = SEM",
    y=1.04,
    fontsize=15,
)
savefig(fig, "colordev_abs_vs_mediancolor_hist2d")
plt.show()

### 8.4  Diagnostic: distribution of `dt_days`, per color pair

Sanity-check figure for section 8.2: histogram of the visit time gap
`dt_days` actually achieved for each color pair, log-scaled x-axis. A
distribution piled up near zero means most color estimates are trustworthy
near-simultaneous pairs; a flat or long-tailed distribution means the
`8.2` panels are dominated by faded, high-`dt_days` points and the
`MAX_DT_DAYS` / `DT_WEIGHT_SCALE_DAYS` choices in the Configuration cell
may need revisiting.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(26, 4), sharey=False)

for ax, color_name in zip(axes, COLOR_NAMES):
    sub = df_color_pairs.loc[df_color_pairs["color_name"] == color_name]
    vals = sub["dt_days"].to_numpy()
    vals = vals[np.isfinite(vals) & (vals > 0)]
    if len(vals) == 0:
        ax.set_title(f"{color_name} (no data)")
        continue

    bins = np.logspace(np.log10(max(vals.min(), 1e-4)), np.log10(vals.max()), 30)
    ax.hist(vals, bins=bins, color="slategray", alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.axvline(np.median(vals), color="crimson", ls="--", lw=1.5, label=f"median={np.median(vals):.2f} d")
    ax.set_xscale("log")
    ax.set_xlabel(r"$\Delta t$ (days)")
    ax.set_title(color_name)
    ax.grid(True, alpha=0.2, which="both")
    ax.legend(fontsize=8)

axes[0].set_ylabel("N color estimates")
fig.suptitle("Distribution of the visit time gap used to build each color estimate", y=1.03, fontsize=14)
plt.tight_layout()
savefig(fig, "dt_days_distribution_per_colorpair")
plt.show()

### 8.6 Want to see a time dependance of the fluctuation excess

For each color pair, one standalone figure (single subplot) showing whether the average absolute color deviation depends on time:

- pairs are grouped by observing night (`day_obs1`, from `match_color_pairs`);
- within each color pair, stars are further split into `N_COLOR_SLICES` tertiles of their own weighted-median color (`color_median_w`), so a genuine night-to-night effect (e.g. driven by PWV or aerosol content on a given night) can be told apart from a color-slice effect;
- each point is the mean `|color deviation|` (mmag) for one (night, color-slice) bin with at least `MIN_PAIRS_PER_NIGHT_SLICE` pairs; the error bar is the **standard error on that mean** (`std / sqrt(n)`), not the RMS spread -- the RMS would be dominated by star-to-star scatter within the slice and would hide any night-to-night effect;
- the bottom x-axis is `expMidptMJD` (mean of the night's points); a secondary top x-axis shows the corresponding calendar date (`YYYY-MM-DD`).

In [ ]:
# one standalone figure per color pair, grouped by observing night and by
# intrinsic-color slice
slice_cmap = plt.get_cmap("viridis")
slice_colors = [slice_cmap(t) for t in np.linspace(0.15, 0.85, N_COLOR_SLICES)]

for color_name in COLOR_NAMES:
    sub = df_color_pairs.loc[df_color_pairs["color_name"] == color_name].copy()
    if len(sub) == 0:
        log.warning("No pairs for %s, skipping section 8.6 figure.", color_name)
        continue

    sub["abs_color_dev_mmag"] = sub["color_dev_mmag"].abs()

    # intrinsic-color slices: tertiles of this star's own weighted-median color
    try:
        sub["color_slice"] = pd.qcut(sub["color_median_w"], N_COLOR_SLICES, labels=False, duplicates="drop")
    except ValueError:
        sub["color_slice"] = 0
    n_slices_here = int(sub["color_slice"].nunique())

    fig, ax = plt.subplots(figsize=(12, 4))
    any_plotted = False

    for s in range(n_slices_here):
        sub_s = sub.loc[sub["color_slice"] == s]
        if len(sub_s) == 0:
            continue
        lo, hi = sub_s["color_median_w"].min(), sub_s["color_median_w"].max()

        # group by observing night (day_obs1), require MIN_PAIRS_PER_NIGHT_SLICE
        night_mjd, night_mean, night_sem = [], [], []
        for _, g in sub_s.groupby("day_obs1"):
            if len(g) < MIN_PAIRS_PER_NIGHT_SLICE:
                continue
            vals = g["abs_color_dev_mmag"].to_numpy()
            night_mjd.append(g["mjd_mean"].mean())
            night_mean.append(vals.mean())
            # standard error on the mean of the absolute fluctuation (NOT the RMS spread)
            night_sem.append(vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0)

        if len(night_mjd) == 0:
            continue

        order = np.argsort(night_mjd)
        night_mjd = np.asarray(night_mjd)[order]
        night_mean = np.asarray(night_mean)[order]
        night_sem = np.asarray(night_sem)[order]

        ax.errorbar(
            night_mjd,
            night_mean,
            yerr=night_sem,
            fmt="o-",
            ms=4,
            lw=1.2,
            capsize=2,
            color=slice_colors[s],
            label=f"{color_name} in [{lo:.2f}, {hi:.2f}] mag  (N nights={len(night_mjd)})",
        )
        any_plotted = True

    if not any_plotted:
        plt.close(fig)
        log.warning(
            "No night has >= %d pairs for %s, skipping section 8.6 figure.",
            MIN_PAIRS_PER_NIGHT_SLICE,
            color_name,
        )
        continue

    ax.set_xlabel("expMidptMJD (night mean)")
    ax.set_ylabel("mean |color deviation| (mmag)\nerror bars = SEM")
    ax.set_title(f"{color_name}: night-by-night color dispersion vs. time")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, title="intrinsic-color slice", loc="best")

    # secondary top x-axis: calendar date, same range as the bottom MJD axis
    xlim = ax.get_xlim()
    ax_top = ax.twiny()
    ax_top.set_xlim(xlim)
    tick_mjd = np.linspace(xlim[0], xlim[1], 6)
    tick_dates = Time(tick_mjd, format="mjd").to_value("iso", subfmt="date")
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_dates, fontsize=8, rotation=20, ha="left")

    plt.tight_layout()
    savefig(fig, f"colordev_vs_time_by_night_{color_name.replace('-', '')}")
    plt.show()

## 9. Summary table export

Save `df_color_summary` (weighted median color + weighted dispersion, one
row per star per color pair) and `df_color_pairs` (every individual color
estimate, with its deviation from the star's weighted median) next to the
figures, for reuse in later notebooks (e.g. a joint u-g vs. z-y comparison
against PWV/aerosol time series).

In [ ]:
out_summary_csv = os.path.join(DIR_FIGS, "colorstats_stablestars_summary.csv")
df_color_summary.to_csv(out_summary_csv, index=False)
log.info("Summary table saved: %s (%d rows)", out_summary_csv, len(df_color_summary))

out_pairs_csv = os.path.join(DIR_FIGS, "colorstats_stablestars_pairs.csv")
df_color_pairs.to_csv(out_pairs_csv, index=False)
log.info("Per-estimate table saved: %s (%d rows)", out_pairs_csv, len(df_color_pairs))